In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import sys
sys.path.append("../")

In [5]:
from optimization_research.ml_constraints import MulticlassSolver
from ortools.linear_solver import pywraplp
from ortools.sat.python import cp_model

In [6]:
import numpy as np
import pandas as pd

In [7]:
# N = 10000 # Number of users
# M = 100 # Number of providers


# solver = pywraplp.Solver.CreateSolver("SAT")

# values = np.random.rand(N) # Weight for each user
# W = (np.random.uniform(0, 1, (N, M)) <= 0.4).astype(int) 

# limits = np.random.uniform(0, 1000, M).astype(int) #Shape of m


# R = np.empty(N).tolist() #The results is a vector
# for i in range(N):
#     R[i] = solver.IntVar(0, 1, 'R{}'.format(i))
# R = np.array(R)    

# demand = (R[:, None] * W).sum(axis=0) #Shape of m (number of providers)
# for i in range(M):
#     solver.Add(demand[i] <= limits[i])

# # Add constraints

# objective_function = (R * values).sum()
# solver.Maximize(objective_function)
# solver.set_time_limit(20000) #Does it work?

# status = solver.Solve()
import numpy as np
import pandas as pd


In [127]:
from typing import List, Dict, Union, Optional, Tuple

class ItayChenSolver:
    def __init__(
            self, 
            input_dataframe: pd.DataFrame,
            limits: dict[str|int], #limits for each provider, if provider not specified, it is unlimited
            values: Optional[np.array] = {}, #values for each user, if not given, all users are equal
            ):
        
        self.effective_users = input_dataframe[input_dataframe["provider"].isin(limits.keys())]["user"].unique()
        self.effective_providers = input_dataframe[input_dataframe["provider"].isin(limits.keys())]["provider"].unique()
        self.limits = limits
        self.users = users
        
        #For the solver, not for the finalized solution.
        self.N = len(self.effective_users)
        self.M = len(self.effective_providers)


        self.values = pd.Series(values, name="values")
        effective_W = input_dataframe[(input_dataframe["user"].isin(self.effective_users)) & (input_dataframe["provider"].isin(self.effective_providers))]
        effective_W = effective_W.groupby(["user", "provider"]).size().unstack(fill_value=0)
        self.effective_W =  effective_W.reindex(self.effective_users, axis=0).reindex(self.effective_providers, axis=1)
        self.output_df = input_dataframe[["user"]].drop_duplicates("user").set_index("user").join(self.values).fillna(1)
        self.effective_values = self.output_df["values"].reindex(self.effective_users)
        

        self.solver = pywraplp.Solver.CreateSolver("SAT")

    def solve(self):
        R = np.empty(self.N).tolist()
        for i in range(self.N):
            R[i] = self.solver.IntVar(0, 1, 'R{}'.format(i))
        R = np.array(R)

        demand = (R[:, None] * self.effective_W.values).sum(axis=0)
        for j in range(self.M):
            self.solver.Add(demand[j] <= self.limits[self.effective_providers[j]])

        objective_function = (R * self.effective_values.values).sum()
        self.solver.Maximize(objective_function)
        status = self.solver.Solve()

        results = pd.Series(R, index = self.effective_users).map(lambda x: x.solution_value())

        if status == self.solver.OPTIMAL:
            print("The problem has an optimal solution")
        
        if status == self.solver.INFEASIBLE:
            print("The problem doesn't have a feasible solution")

        self.output_df["decisions"] = results
        self.output_df["decisions"] = self.output_df["decisions"].fillna(1)
        
    
users = ["user1", "user2", "user3", "user4", "user5"]
providers = ["provider1", "provider2", "provider3", "provider4"]
limits = {"provider1": 2, "provider2": 200, "provider777":12}


#pd.Series(W).unstack().loc["user1"].reindex(limits.keys())#.isna().sum()

input_dataframe = pd.DataFrame([
    ["user1", "credential1", "provider1"], 
    ["user1", "credential2", "provider1"], 
    ["user1", "credential3", "provider2"], 
    ["user3", "credential4", "provider3"], 
    ["user4", "credential5", "provider3"],
    ["user5", "credential6", "provider1"],
    ], columns = ["user", "credential", "provider"])



ics = ItayChenSolver(input_dataframe, limits = limits, values = {"user1":100})
ics.effective_values
# a.W

input_dataframe

ics.solve()

The problem has an optimal solution


In [129]:
ics.effective_W

provider,provider1,provider2
user,,
user1,2,1
user5,1,0


In [115]:
ics.solver.variable("R1")

TypeError: in method 'Solver_variable', argument 2 of type 'int'

In [ ]:
values = pd.Series(values, name="values")
decisions = pd.Series([1, 0], name="decision", index = effective_users)

output_dataframe = input_dataframe[["user"]].drop_duplicates("user").set_index("user").join(values).fillna(1)
output_dataframe.join(decisions).fillna(1)

effective_values = 
effective_values

In [84]:
values = {"user1": 10, "user2": 2, "user3": 3, "user9":100}
effective_values = pd.Series(values).reindex(effective_users, fill_value=1) #TODO
effective_values


#in_our_pocket_users = [user for user in values.keys() if user not in effective_users]

['user2', 'user3', 'user9']

In [92]:
effective_users

array(['user1', 'user5'], dtype=object)

user
user1    10.0
user5     1.0
Name: values, dtype: float64

In [79]:


input_dataframe[(~input_dataframe["user"].isin(effective_users)) & (~input_dataframe["user"].isin(values.keys()))]["user"].nunique()

2

In [60]:
effective_users = input_dataframe[input_dataframe["provider"].isin(limits.keys())]["user"].unique()
effective_providers = input_dataframe[input_dataframe["provider"].isin(limits.keys())]["provider"].unique()
effective_W = input_dataframe[(input_dataframe["user"].isin(effective_users)) & (input_dataframe["provider"].isin(effective_providers))]
effective_W = effective_W.groupby(["user", "provider"]).size().unstack(fill_value=0)



provider,provider1,provider2
user,,
user1,2,1
user5,0,1


In [55]:
effective_W

,user,credential,provider
0,user1,credential1,provider1
1,user1,credential2,provider1
2,user1,credential3,provider2
5,user5,credential6,provider2


In [37]:
limits.keys()

dict_keys(['provider1', 'provider2', 'provider777'])

In [15]:
#create 10000000 by 1000 sparse matrix
#randomly select 1000 rows

# a = np.random.rand(10000000, 1000)

In [ ]:
W

In [ ]:
 #If not specified, the limit is infinite

In [74]:
results = pd.Series(R).map(lambda x: x.solution_value()).values#[:, 1]

In [76]:
results.mean()

0.0359

In [31]:
n = 4
m = 3

X = np.array([1,1,1,1]) # input
W = np.array([[1,2,1],
              [0,1,0],
              [1,0,1],
              [0,0,1]]) # weights

value = np.array([1,2,3,4]) # value of each item



demand = (X[:, None] * W).sum(axis=0)
all(demand<=limits) #we are limited to this
(value*X).sum() #We should maximize this value

10